In [17]:
# Enable autoreload when modifying files
%load_ext autoreload
%autoreload 2

In [42]:
# -------------------------------------------
# Load dataset
# -------------------------------------------
# Load database
import pandas as pd
from pathlib import Path

# 2. Get Real Data Directory
dir = Path('/app/notebooks/data/synthetic/notebook_test/')
print(f"Using data from: {dir.name}")

# 3. Load Base Data
df_problems = pd.read_csv(dir / 'icare_problems_anon.csv')

n_codes = df_problems.PROBLEM_CODE.nunique()
n_desc = df_problems.PROBLEM_DESC.nunique()
print(f"DataFrame {df_problems.shape} | n_codes: {n_codes} | n_desc: {n_desc}:")
display(df_problems.head(3))

Using data from: notebook_test
DataFrame (303, 6) | n_codes: 12 | n_desc: 13:


,SUBJECT,ENCNTR_ID,PROBLEM_CODE,PROBLEM_DESC,PROBLEM_DT_TM,UPDATE_DT_TM
0,10001,8600986,38341003,hypertension,2023-05-28,2023-08-23
1,10001,3600304,84114007,heart failure,2021-12-06,2022-01-21
2,10001,4451560,40930008,hypothyroidism,2021-07-11,2021-08-28


In [40]:
# ----------------------------------------------------------
# Compute history efficient
# ----------------------------------------------------------
# Libraries
import yaml

from icare_risk.clinphen.utils.history import build_historical_events_table
from icare_risk.clinphen.utils.history import evaluate_temporal_phenotypes

sample_config = {
    'has_diabetes': {
        'kwargs': {
            'codes': [ '44054006', '73211009', '46635009' ]
        }
    }
}

# Load file
yaml_path = '/app/src/icare_risk/config/icare/phenotypes.yaml'
with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

# Select Charlson phenotype entries
config = {k:v for k,v in config.items() if  k.startswith('charlson_')}

# Create auxiliary dataframe with first occurence
df_first_occ = build_historical_events_table(
    df=df_problems,
    configs=config,
    subject_col='SUBJECT',
    time_col='PROBLEM_DT_TM',
    code_col='PROBLEM_CODE'
)
display(df_first_occ.head(5))

# Count number of patients per category
result = df_first_occ.groupby('event_name')['SUBJECT'] \
    .nunique().reset_index(name='patient_count')
display(result)


,SUBJECT,event_name,first_occurrence_date
0,10001,charlson_hx_chf,2021-12-06
1,10001,charlson_hx_diabetes_uncomp,2020-08-25
2,10003,charlson_hx_diabetes_uncomp,2023-06-22
3,10003,charlson_hx_pulmonary,2023-01-19
4,10004,charlson_hx_diabetes_uncomp,2020-10-26


,event_name,patient_count
0,charlson_hx_chf,21
1,charlson_hx_diabetes_uncomp,48
2,charlson_hx_pulmonary,24
3,charlson_hx_renal_mod_sev,15
